# 06. Concurrency: the GIL, threads, processes and asyncio

**Goal.** Pick the right concurrency model for a workload from measured evidence, not folklore.

**You will be able to:**
1. Explain what the Global Interpreter Lock protects and what it does not.
2. Show with timings that threads help I/O-bound work and not CPU-bound work.
3. Use processes for CPU-bound work and account for start-up and pickling costs.
4. Use `asyncio` for many concurrent waits and avoid blocking the event loop.

Timings come from your machine. Read the ratios, not the absolute numbers. Worker functions live in `understanding_python.concurrency` so child processes can import them on Windows and macOS (the `spawn` start method).

## 1. The GIL: what it is and what it costs

### How it works under the hood
- CPython's memory management is not thread-safe: reference counts change without atomic operations. The **Global Interpreter Lock** lets one thread run Python bytecode at a time.
- A running thread releases the lock when it blocks on I/O or `time.sleep`.
- It also releases the lock after the **switch interval** (`sys.getswitchinterval()`, 5 ms by default) when another thread asks for it.
- The GIL guarantees that single bytecodes do not corrupt interpreter state. It does not make `counter += 1` atomic: that is a load, an add and a store, and a switch can land between them.
- Python 3.13 has an experimental free-threaded build with no GIL. This notebook runs on the standard build. `sys._is_gil_enabled()` tells you which one you have.
- C extensions (NumPy, hashlib, zlib) can release the GIL during long native calls, so threads can scale there.

### Production use
- Web servers use threads or processes per request. Gunicorn runs several worker processes so each has its own GIL.
- Data libraries parallelise inside native code. Pure Python loops stay on one core per process.

### Trade-offs

| Aspect | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| GIL (standard build) | Small, shared heap | Fast single-thread code (no per-object locks) | Simple mental model | Safe interpreter, fast single thread | CPU-bound threads do not scale |
| Free-threaded build (3.13 experimental) | Larger per-object overhead | Slower single thread today | Same API | True thread parallelism | Extensions must be rebuilt, still maturing |
| Native-code parallelism | Shared arrays | Releases the GIL | Hidden in the library | Scales on cores | Only inside the library |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import dis
import os
import sys
import time

from understanding_python import concurrency as cc

print("GIL enabled          :", sys._is_gil_enabled())
print("switch interval (s)  :", sys.getswitchinterval())
print("logical CPUs         :", os.cpu_count())


def bump(counter):
    counter.value += 1


dis.dis(bump)

N = 600_000
CHUNKS = cc.split_range(0, N, 16)  # more chunks than workers balances the uneven cost of larger numbers
expected = cc.count_primes((0, N))
print("primes below", f"{N:,}", ":", f"{expected:,}")

results = {}
_, results["sequential"] = cc.timed(lambda: cc.run_sequential(cc.count_primes, CHUNKS))
out, results["4 threads"] = cc.timed(lambda: cc.run_threads(cc.count_primes, CHUNKS, workers=4))
assert sum(out) == expected
print(f"sequential {results['sequential']:.2f} s | 4 threads {results['4 threads']:.2f} s | thread speed-up {results['sequential'] / results['4 threads']:.2f}x")

GIL enabled          : True
switch interval (s)  : 0.005
logical CPUs         : 8
 13           RESUME                   0

 14           LOAD_FAST                0 (counter)
              COPY                     1
              LOAD_ATTR                0 (value)
              LOAD_CONST               1 (1)
              BINARY_OP               13 (+=)
              SWAP                     2
              STORE_ATTR               0 (value)
              RETURN_CONST             0 (None)


primes below 600,000 : 49,098


sequential 1.32 s | 4 threads 1.32 s | thread speed-up 1.00x


## 2. Threads: overlapping waits, and the race you still get

### How it works under the hood
- A `threading.Thread` is a real OS thread. The OS scheduler runs it, and the GIL decides which thread executes bytecode.
- Blocking calls (`time.sleep`, socket reads, file reads, `requests.get`) release the GIL while they wait. Other threads run in that gap, so many waits overlap.
- Shared mutable state needs a lock. `threading.Lock` is a mutex: `with lock:` acquires, runs the critical section and releases.
- `ThreadPoolExecutor` keeps a fixed set of worker threads fed from a queue, which avoids creating a thread per task.

### Production use
- Calling many HTTP APIs, reading many files or running many database queries from a synchronous codebase.
- Background workers that wait on queues. Keep shared state small and guarded.

### Trade-offs

| Aspect | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Threads for I/O | About 8 MB reserved stack per thread (virtual) | Waits overlap, total near the slowest call | Ordinary blocking code | Easy to add to existing code | Race conditions, no CPU parallelism |
| Locks | Tiny | Contention serialises work | Critical sections are explicit | Correct shared state | Deadlocks, forgotten locks |
| Sequential | Lowest | Sum of all waits | Simplest | No concurrency bugs | Slow for I/O |

In [3]:
calls = [0.2] * 8
_, seq_io = cc.timed(lambda: cc.run_sequential(cc.blocking_io, calls))
_, thr_io = cc.timed(lambda: cc.run_threads(cc.blocking_io, calls, workers=8))
results["io sequential"], results["io threads"] = seq_io, thr_io
print(f"8 calls of 200 ms: sequential {seq_io:.2f} s | 8 threads {thr_io:.2f} s | speed-up {seq_io / thr_io:.1f}x")

threads, times = 4, 5_000
total = threads * times
plain = cc.hammer(threads, times, locked=False)
print(f"plain read-add-write: {plain:,} of {total:,} (3.13 rarely switches inside such a short window)")
racy = cc.hammer(threads, times, locked=False, pause=True)
print(f"same code with one yielding call in the window: {racy:,} of {total:,}, lost {total - racy:,}")
safe = cc.hammer(threads, times, locked=True, pause=True)
print(f"with a Lock around the window: {safe:,} of {total:,}")

8 calls of 200 ms: sequential 1.60 s | 8 threads 0.20 s | speed-up 7.9x
plain read-add-write: 20,000 of 20,000 (3.13 rarely switches inside such a short window)


same code with one yielding call in the window: 5,051 of 20,000, lost 14,949


with a Lock around the window: 20,000 of 20,000


## 3. Processes: real parallelism with a price

### How it works under the hood
- `multiprocessing` starts separate interpreter processes, each with its own GIL and heap. They run on different cores at once.
- On Windows and macOS the start method is `spawn`: Python launches a fresh interpreter, imports your module and runs the target. That is why workers must be importable functions, not notebook-local ones.
- Arguments and results travel between processes through `pickle` over pipes. Large objects are copied, not shared.
- `ProcessPoolExecutor` reuses workers across tasks, so you pay start-up once per pool.
- Shared memory (`multiprocessing.shared_memory`) and memory-mapped files avoid copying large arrays.

### Production use
- CPU-bound batch work: image resizing, feature extraction, report generation, parsing large files.
- Web servers: one worker process per core (Gunicorn, Uvicorn workers).
- Data loaders in machine-learning training use worker processes to prepare batches.

### Trade-offs

| Aspect | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Process pool | One interpreter per worker (tens of MB) | Start-up 100 ms or more, pickling per task | Same `map` API as threads | Scales with cores, isolates crashes | Costly for small tasks, no shared objects |
| Threads | Shared heap | Cheap to start | Same API | Light | Do not speed up CPU-bound code |
| Native vectorisation | Shared arrays | No Python loop | Different style | Often fastest | Only for numeric patterns |

In [4]:
import multiprocessing as mp
import pickle

print("start method:", mp.get_start_method())

t0 = time.perf_counter()
out, results["4 processes"] = cc.timed(lambda: cc.run_processes(cc.count_primes, CHUNKS, workers=4))
assert sum(out) == expected
print(f"4 processes {results['4 processes']:.2f} s (includes pool start-up) | speed-up over sequential {results['sequential'] / results['4 processes']:.2f}x")

small = [(0, 50)] * 4
_, small_seq = cc.timed(lambda: cc.run_sequential(cc.count_primes, small))
_, small_proc = cc.timed(lambda: cc.run_processes(cc.count_primes, small, workers=4))
print(f"tiny tasks: sequential {small_seq * 1000:.2f} ms | processes {small_proc * 1000:.0f} ms (start-up dominates)")

payload = list(range(1_000_000))
print(f"pickled size of a 1M-int list: {len(pickle.dumps(payload)) / 1e6:.1f} MB crosses the pipe per task")

start method: spawn


4 processes 0.73 s (includes pool start-up) | speed-up over sequential 1.82x


tiny tasks: sequential 0.07 ms | processes 302 ms (start-up dominates)
pickled size of a 1M-int list: 4.9 MB crosses the pipe per task


## 4. `asyncio`: many waits on one thread

### How it works under the hood
- A coroutine function (`async def`) returns a coroutine object, a close relative of a generator. `await` suspends it and returns control to the **event loop**.
- The loop keeps a ready queue and asks the OS selector (`epoll`, `kqueue`, IOCP) which sockets are ready. It resumes coroutines whose awaited event finished.
- Everything runs on one thread, so there are no data races between `await` points. A coroutine that never awaits blocks every other task.
- Jupyter already runs a loop, so notebooks use top-level `await`. Scripts call `asyncio.run(main())`.
- `asyncio.gather` runs awaitables concurrently. `asyncio.Semaphore` caps how many run at once. `asyncio.to_thread` moves blocking calls to a worker thread.

### Production use
- API gateways, chat servers and scrapers that hold thousands of open connections.
- FastAPI endpoints that await database drivers and HTTP clients.
- Rate-limited fan-out: fetch 10,000 URLs, 50 at a time.

### Trade-offs

| Aspect | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| `asyncio` | A few KB per task | Overlaps thousands of waits | `async`/`await` colouring spreads through the code | Cheapest per connection, no locks between awaits | One blocking call stalls everything; needs async libraries |
| Threads | MBs per thread | Overlaps tens to hundreds of waits | Plain blocking code | Works with any library | Heavier, needs locks |
| Processes | Tens of MB each | Start-up cost | Plain code | CPU parallelism | Not for thousands of waits |

In [5]:
# Jupyter already runs an event loop, so use top-level await here. Scripts call asyncio.run(main()).
import asyncio

delays = [0.1] * 200
t0 = time.perf_counter()
done = await cc.gather_io(delays)
unbounded = time.perf_counter() - t0
t0 = time.perf_counter()
done = await cc.gather_io(delays, limit=50)
limited = time.perf_counter() - t0
print(f"200 waits of 100 ms: all at once {unbounded:.2f} s | semaphore of 50 {limited:.2f} s (4 waves) | sequential would take {sum(delays):.0f} s")

blocked = await cc.ticks_during(cc.sleep_blocking_100ms, offload=False)
free = await cc.ticks_during(cc.sleep_blocking_100ms, offload=True)
print(f"loop wake-ups during a 100 ms blocking call: called directly {blocked} | via to_thread {free}")

coro = cc.async_io(0.01)
print("a coroutine object:", type(coro).__name__, "| frame:", coro.cr_frame is not None)
await asyncio.wait_for(coro, timeout=1)
try:
    await asyncio.wait_for(cc.async_io(1.0), timeout=0.05)
except TimeoutError:
    print("wait_for cancelled the slow task after 50 ms")

200 waits of 100 ms: all at once 0.12 s | semaphore of 50 0.44 s (4 waves) | sequential would take 20 s
loop wake-ups during a 100 ms blocking call: called directly 1 | via to_thread 6
a coroutine object: coroutine | frame: True


wait_for cancelled the slow task after 50 ms


## 5. Choosing a model from the evidence

| Workload | Best fit | Why |
|---|---|---|
| Thousands of network waits | `asyncio` | Cheapest per connection |
| Tens of blocking calls in sync code | Thread pool | Waits overlap, no rewrite |
| CPU-bound pure Python | Process pool | Separate GILs |
| Numeric arrays | NumPy or native code | Releases the GIL, uses SIMD |
| Mixed | Async front end with `to_thread` or a process pool for heavy parts | Each part uses its own strength |

In [6]:
width = max(len(k) for k in results)
print(f"{'workload':<{width}}  seconds  vs baseline")
for name, base in (("sequential", "sequential"), ("4 threads", "sequential"), ("4 processes", "sequential"),
                   ("io sequential", "io sequential"), ("io threads", "io sequential")):
    print(f"{name:<{width}}  {results[name]:7.2f}  {results[base] / results[name]:5.2f}x")

workload       seconds  vs baseline
sequential        1.32   1.00x
4 threads         1.32   1.00x
4 processes       0.73   1.82x
io sequential     1.60   1.00x
io threads        0.20   7.87x
